PRESENTAZIONE

#1 NICK

Introduzione + regole + funzioni di animazione -> (punto 1 + 2)

#2 KEVIN

Funzione pattern + lista dei pattern

#3 ENRICA

GoL + Freq + distr conf

#4 ETTORE

Time to stab + Occupanza media

| **Name** | **Family Name** | **Matriculation Number** |
| -------- | --------------- | ------------------------ |
| Nicolò   | Meneguzzo       | *******                  |
| Kevin    | Brugnera        | 2196578                  |
| Enrica   | Calabretto      | 2198447                  |
| Ettore   | Vissa           | 2197527                  |


## Introduction

[Game of Life](http://en.wikipedia.org/wiki/Conway's_Game_of_Life) (GoF) is a cellular automaton devised by the British mathematician John Horton Conway in 1970. The game is a zero-player game, meaning that its evolution is determined by its initial state, requiring no further input. One interacts with the Game of Life by creating an initial configuration and observing how it evolves, or, for advanced players, by creating patterns with particular properties.


The universe of the Game of Life is an infinite two-dimensional orthogonal grid of square cells, each of which is in one of two possible states, live or dead. Every cell interacts with its eight neighbours, which are the cells that are directly horizontally, vertically, or diagonally adjacent. At each step in time, the following transitions occur:

* Any live cell with fewer than two live neighbours dies, as if by needs caused by underpopulation.
* Any live cell with more than three live neighbours dies, as if by overcrowding.
* Any live cell with two or three live neighbours lives, unchanged, to the next generation.
* Any dead cell with exactly three live neighbours becomes a live cell.

The initial pattern constitutes the 'seed' of the system. The first generation is created by applying the above rules simultaneously to every cell in the seed – births and deaths happen simultaneously, and the discrete moment at which this happens is sometimes called a tick. (In other words, each generation is a pure function of the one before.) The rules continue to be applied repeatedly to create further generations.

---
# Start off implementing the GoF's rules and play with simple seeds in small dimensions

In [2]:
# Calcolo numerico
import numpy as np
import numpy.random as npr
from scipy.signal import convolve2d
import pandas as pd 

# Visualizzaione
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100.0
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.animation import FuncAnimation, PillowWriter

# Ambiente interattivo e display delle animazioni
from IPython.display import display, Markdown, Image
from matplotlib.animation import FuncAnimation, PillowWriter

In [3]:
def rule(grid):
    new_grid = np.zeros_like(grid)
    
    filter_neighborhood = np.array([[1,1,1],
                                    [1,0,1],
                                    [1,1,1]])

    # Convolution of the filter
    neighbors = convolve2d(grid, filter_neighborhood, mode='same', boundary='fill', fillvalue=0)

    # Rule masks
    rule_birth = (neighbors == 3) # every cell with exactly 3 neighbors 
    rule_death = (grid == 1) & ((neighbors < 2) | (neighbors > 3)) # alive cell with under/over crowding condition
    rule_survive = (grid == 1) & (neighbors == 2) # alive cell with exactly 2 neighbors 

    # Updates
    new_grid[rule_birth] = 1
    new_grid[rule_death] = 0
    new_grid[rule_survive] = 1

    return new_grid

def initialize(grid_size, pattern):
    grid = np.zeros(grid_size) 
    sx = (grid_size[0] - pattern.shape[0]) // 2
    sy = (grid_size[1] - pattern.shape[1]) // 2
    
    grid[sx:sx + pattern.shape[0], sy:sy + pattern.shape[1]] = pattern
    return grid

In [4]:
def game_of_life(pattern, epochs, size, figsize, fps=5):

    # Inizializzazione

    global grid, time
    time = 0
    grid = initialize(size, pattern) 

    # Visualizzazione 

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.imshow(grid, cmap='binary', extent=[0, size[1], 0, size[0]])

    def update(frame):
        global grid, time
        if frame > 0:
            grid = rule(grid) 
            time += 1
        # Aggiornamento dei dati visivi
        img.set_data(grid)
        ax.set_title(f'Game of Life: epoch = {time}')
        return [img]

    # Creazione dell'animazione
    ani = FuncAnimation(fig, update, frames=epochs, interval=100, blit=True)   
    ax.set(title='Game of Life Grid', xticks=np.arange(0, size[1], 1), yticks=np.arange(0, size[0], 1), 
            xticklabels=[],  yticklabels=[], aspect='equal')
    ax.grid(color='gray', linestyle='-', linewidth=1)

    # Salvataggio dell'animazione in formato gif
    ani.save("temp.gif", writer=PillowWriter(fps=fps))
    plt.close(fig)
    
    # Display dell'animazione
    return display(Image("temp.gif"))
    
    # remove per evitare di occupare memoria inutile
    !rm temp.gif 

In [ ]:
pattern = np.array([[1,1,0,0],
                    [0,0,1,1],
                    [0,1,0,1],
                    [1,1,0,1]])

game_of_life(pattern, epochs=35, size=(11, 11), figsize=(6, 6), fps=3)

---
## Increase the size of the GoF's world and play with more advanced pattern

In [5]:
# Evoluzione del numero di celle vive nel tempo

def plot_alive_cells(pattern, epochs, size, figsize=(10, 6), fps=5):

    # Inizializzazione

    global grid, epoch
    epoch = 0
    grid = initialize(size, pattern)
    alive_counts = [np.sum(grid)]

    # Trovo ymax, ymin per definire ylim della figura
    
    tmp = grid.copy()
    alive = np.sum(tmp)
    min_alive, max_alive = alive, alive
    for i in range(epochs):
        tmp = rule(tmp)
        alive = np.sum(tmp)
        max_alive = max(max_alive, alive)
        min_alive = min(min_alive, alive)

    # Visualizzazione

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    # ax1 -> grid evolution
    img = ax1.imshow(grid, cmap='binary', extent=[0, size[1], 0, size[0]])
    ax1.set(title='Game of Life Grid', xticks=np.arange(0, size[1], 1), yticks=np.arange(0, size[0], 1), 
          xticklabels=[],  yticklabels=[], aspect='equal')
    ax1.grid(color='gray', linestyle='-', linewidth=1)
    # ax2 -> trend plot
    line, *_ = ax2.plot(alive_counts, color = 'r', label='Alive Cells')
    

    def update(frame):
        global grid, epoch
        # update
        if frame > 0:
            grid = rule(grid)
            alive_counts.append(int(np.sum(grid)))
            epoch += 1
            
        # Aggiornamento degli assi
        img.set_data(grid)
        ax1.set_title(f'Game of Life: epoch = {epoch}')
        line.set_data(range(len(alive_counts)), alive_counts) # updated frame by frame
        return [img, line]

    # Creazione dell'animazione
    ani = FuncAnimation(fig, update, frames=epochs, interval=100, blit=True)
    
    ax1.set(title='Game of Life Grid', xticks=np.arange(0, size[1], 1), yticks=np.arange(0, size[0], 1), 
            xticklabels=[],  yticklabels=[], aspect='equal')
    ax1.grid(color='gray', linestyle='-', linewidth=1)
    ax2.set(title='Alive Cells Evolving Through Epochs', xlabel='Epochs', ylabel='Number of Alive Cells', 
            xlim=(0, epochs - 1), ylim=([min_alive -2, max_alive + 2]))
    ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2.grid(True, which='major', color='gray', linestyle='--', alpha=0.7)
    ax2.legend(loc='upper right')

    # Salvataggio dell'animazione in formato gif
    ani.save("temp.gif", writer=PillowWriter(fps=fps))
    plt.close(fig)
    
    # Display dell'animazione
    return display(Image("temp.gif"))
    
    # remove per evitare di occupare memoria inutile
    !rm temp.gif 

In [6]:
# Evoluzione dell'occupanza nel tempo

def hist_alive_cells(pattern, epochs, size, figsize=(8, 8), fps=5):

    #Inizializzazione

    global grid, occupancy, epoch
    epoch = 0
    grid = initialize(size, pattern)
    occupancy = grid.copy()

    fig, ax = plt.subplots(figsize=figsize)

    hist = ax.imshow(occupancy, cmap='viridis', vmin=0, vmax=epochs, extent=[0, size[1], size[0], 0])
    ax.set(title='Alive Cells Occupancy Evolving Through Epochs', xticks=np.arange(0, size[1], 1), yticks=np.arange(0, size[0], 1), 
          xticklabels=[],  yticklabels=[], aspect='equal')
    ax.grid(color='gray', linestyle='-', linewidth=1)
    fig.colorbar(hist, label='Cell Occupancy')

    def update(frame):
        global grid, occupancy, epoch
        # update valori
        if frame > 0:
            grid = rule(grid)
            occupancy += grid
            epoch += 1
    
        # update animazione
        hist.set_data(occupancy)
        ax.set_title(f"Alive Cells Cumulative Occupancy at epoch = {epoch}")
        return [hist]

    # Creazionedell'animazione
    ani = FuncAnimation(fig, update, frames=epochs, interval=100, blit=True)

    # Salvataggio dell'animazione in formato gif
    ani.save("temp.gif", writer=PillowWriter(fps=fps))
    plt.close(fig)
    
    # Display dell'animazione
    return display(Image("temp.gif"))
    
    # remove per evitare di occupare memoria inutile
    !rm temp.gif 

Alcuni esempi con pattern random per la visualizzazione

In [ ]:
# pattern random uniforme 
size_grid, epochs = (20, 20), 200
pattern = npr.randint(0, 2, size=size_grid)

plot_alive_cells(pattern, epochs, size_grid, fps=10)
hist_alive_cells(pattern, epochs, size_grid, figsize=(6, 6), fps=15)

# pattern gaussiano (1D)
size_grid, epochs = (40, 40), 200
pattern = np.heaviside( npr.normal(size = size_grid), 0)

plot_alive_cells(pattern, epochs, size_grid, fps=10)
hist_alive_cells(pattern, epochs, size_grid, figsize=(6, 6), fps=15)


# pattern gaussiano (2D)
N = 30
size_grid, epochs = (N, N), 500
rand = np.random.multivariate_normal([N//2, N//2], 10*np.eye(2), size=100)
bins = np.arange(0, N+1)
 
grid_rand = np.column_stack((np.digitize(rand[:, 0], bins) - 1, np.digitize(rand[:, 1], bins) - 1)) # riscala indici

pattern = np.zeros(size_grid)
pattern[grid_rand[:, 0], grid_rand[:, 1]] = 1
plot_alive_cells(pattern, epochs, size_grid, fps=10)
hist_alive_cells(pattern, epochs, size_grid, figsize=(6, 6), fps=15)

In [ ]:
pattern = rle_to_pattern('bakersdozenreactions')
plot_alive_cells(pattern, 100, size=(30, 40), figsize=(18, 8))

---
## Implement examples of the three categories of patterns *still lifes*, *oscillators* and *spaceships*

To reproduce the evolution of different patterns in the Game of Life, we first need to initialize the simulation grid. Manually filling an $m \times n$ grid entry by entry would be extremely time-consuming and highly prone to typing errors.

To simplify this process, we implement a function that allows us to import pre-built patterns.  
These patterns are stored in files using the **`.rle`** (Run-Length Encoding) format, that is a compact representation of patterns that describes consecutive cells with the same state using counts and symbols.

The `.rle` files are downloaded from the repository [Cellular Automata Patterns](https://github.com/thomasdunn/cellular-automata-patterns/tree/master/patterns/conwaylife); our function reads the `.rle` file and translates the encoded pattern into a grid array, which is then used to initialize the simulation. Once the grid is initialized, the pattern can evolve according to the Game of Life rules.

In [1]:
def rle_to_pattern(pattern_name):

    # Download the pattern from github
    url = f'https://raw.githubusercontent.com/thomasdunn/cellular-automata-patterns/refs/heads/master/patterns/conwaylife/{pattern_name}.rle'
    !wget -q {url} # -q flag to avoid output print

    # Save the pattern as a string
    pattern_rle = []
    with open(f'{pattern_name}.rle', mode='r') as file:
        for line in file:
            if (line[0] != '#'): # remove metadata
                pattern_rle.append(line[:-1])  # remove \n
    grid_info = pattern_rle[0] # first line
    pattern_rle = ''.join(pattern_rle[1:])

    # Remove the file
    !rm {pattern_name}.rle

    # Get the pattern's width
    width = ''
    for digit in grid_info[:6]: # first 6 digit 
        if digit.isdigit():
            width += digit
    width = int(width)
    
    # Converting pattern string in a grid
    pattern = []
    line, times = '', ''
    for digit in pattern_rle:

        # repetition number
        if digit.isdigit():
            times += digit
            continue
        if times != '':
            n = int(times)
            times = ''
        else:
            n = 1

        # rule $
        if digit == '$':
            for i in range(n):
                diff = width - len(line)
                if diff > 0:
                    line += '0' * diff
                pattern.append(line) 
                line = ''
            continue
        # death cells
        if digit == 'b':
            for i in range(n):
                line += '0'
        # alive cells
        elif digit == 'o':  
            for i in range(n):
                line += '1'
        # end of the pattern
        elif digit == '!':  
            break

    # last line
    if len(line) != 0:
        diff = width - len(line)
        if diff > 0:
            line += '0' * diff
        pattern.append(line)

    # to array
    grid = np.array([[int(i) for i in line] for line in pattern])
    return grid

Below is a simple example of a pattern encoded using the `.rle` format:

In [7]:
rle_lines = ["b2o3b$ ","o2bo2b$","b2obob$","2bobob$","obob2o$","2o!    "]
pattern = rle_to_pattern('beehat')

for rle, row in zip(rle_lines, pattern):
    print(f"{rle} -> {row}")

b2o3b$  -> [0 1 1 0 0 0]
o2bo2b$ -> [1 0 0 1 0 0]
b2obob$ -> [0 1 1 0 1 0]
2bobob$ -> [0 0 1 0 1 0]
obob2o$ -> [1 0 1 0 1 1]
2o!     -> [1 1 0 0 0 0]


In Conway’s Game of Life, patterns are classified according to their dynamical behavior during evolution. The three main categories are:

1. **Still Lifes**
2. **Oscillators**
3. **Spaceships**

These three categories represent the most basic types of objects in the Game of Life and form the building blocks of all possible patterns that can emerge in the game.

Each category can be distinguished by its specific behavior during the evolution from one epoch to the next. Below follows a description of these categories, along with some examples.

## 1. *Still Lifes*

A ***still life*** is a pattern that remains unchanged from one generation to the next.  
For this reason, these patterns are the simplest possible structures in the Game of Life: from one epoch to another they do not evolve or move, remaining exactly the same configuration.

According to the rules of the Game of Life, the minimum number of live cells required to form a *still life* is **four**.  
In this case, there are only two different patterns: the **block** and the **tub**.

In [ ]:
block = rle_to_pattern('block')
tub = rle_to_pattern('tub')

display(Markdown("### **Block pattern**"))
game_of_life(block, epochs=50, size=(6, 6), figsize=(3, 3))
display(Markdown("### **Tub pattern**"))
game_of_life(tub, epochs=50, size=(5, 5), figsize=(3, 3))

For configurations with more than seven live cells, many *still life* patterns can be found. In this case, we can distinguish *pseudo-still lifes*, which are composed of smaller, disconnected patterns placed next to each other. These are generally considered less relevant than standard connected *still lifes*, some examples of which are reported below.

In [ ]:
mickeymouse = rle_to_pattern('mickeymouse')
triplepseudostilllife = rle_to_pattern("triplepseudostilllife")

display(Markdown("### **Mickeymouse pattern**"))
game_of_life(mickeymouse, epochs=50, size=(11, 12), figsize=(4, 4))
display(Markdown("### **Triple pseudo-still life pattern**"))
game_of_life(triplepseudostilllife, epochs=50, size=(12, 12), figsize=(4, 4))

## 2. *Oscillators*

An ***oscillator*** is a pattern that cycles through a finite number of distinct configurations. The configurations observed at each epoch are called **phases**. The smallest number of epochs required for the pattern to return to its initial phase is defined as the **period** of the oscillator.

The minimum possible period is 2. The oscillator with the smallest number of live cells is the **blinker**: it is extremely common in random initial configurations and often emerges from small chaotic clusters.

For larger periods or a greater number of live cells, many more complex *oscillators* can be found. Some examples are shown below.

In [ ]:
blinker = rle_to_pattern('blinker')
pentadecathlon = rle_to_pattern("pentadecathlon")

display(Markdown("### **Blinker pattern**"))
plot_alive_cells(blinker, epochs=20, size=(5, 5), figsize=(10, 5))
display(Markdown("### **Pentadecathlon pattern**"))
plot_alive_cells(pentadecathlon, epochs=60, size=(13, 16))

Also, the **Pentadecathlon** is often generated by random initial configurations and has a period of 15. Its frequent appearance in random initial states makes it one of the most naturally occurring higher-period oscillators.

In [ ]:
koksgalaxy = rle_to_pattern("koksgalaxy")

display(Markdown("### **Kok’s Galaxy  pattern**"))
plot_alive_cells(koksgalaxy, epochs=40, size=(17, 17), figsize=(18, 8))

In contrast, **Kok’s Galaxy** has a period of 8 epochs and consists of two interlocking rotating structures. During its evolution, parts of the pattern appear to rotate around a central axis while preserving global symmetry. Unlike the Pentadecathlon, it is rarely produced spontaneously due to his structural complexity.

## 3. *Spaceships*

Different from the previous categories, these patterns are not stationary and produce moving objects. In particular, *spaceships* return to their initial phase after a finite number of generations, defining their **period**, but at a different location from where they started.  
Since they are moving objects, a **velocity** can also be defined, measured as the number of cells traveled per generation.  

The most common *spaceship* is the **Glider**, which is frequently produced by larger patterns and moves across the grid while the initial pattern continues to evolve. The first known glider generator is the **Gosper glider gun**, shown below.


In [ ]:
glider = rle_to_pattern('glider')
gosperglidergun = rle_to_pattern("gosperglidergun")

display(Markdown("### **Glider pattern**"))
game_of_life(glider, epochs=15, size=(10, 10), figsize=(4, 4))
display(Markdown("### **Gosper Glider Gun pattern**"))
game_of_life(gosperglidergun, epochs=100, size=(30, 40), figsize=(10, 6), fps=10)

---
## Analyse the evolutions of the patterns in terms of frequency, replication, occupancy, etc.

Analisi del periodo e della frequenza

In [ ]:
# funzione veloce per ritornare il gli stati

def GoL_simulation(pattern, epochs, size):

    # GoL simulation and collection of pattern at each epoch
    
    grid = initialize(size, pattern) # initial state
    states = [grid]
    
    for t in range(1, epochs):
        new_grid = rule(grid)
        grid= new_grid
        states.append(new_grid)

    return states

The funciotn `analyze_frequency` gives the period( in epoch), after it has find two identical pattern in a grid with two consecutive states: in the case of the still life and the pattern that extinct themselves the period is 1, since the function works on the last two states, that are identical. Instead if the pattern is chaotic (so it doesn't reach the stability after the epoch that we choose) the function return None and print 'No global repetition'.

In [ ]:
def analyze_frequency(pattern, epochs, size, verbose = False):
    grid = initialize(size, pattern)
    grid_history = {}  # dizionario: bytes -> epoch
    
    for epoch in range(epochs):
        key = grid.tobytes()  # converte la griglia in bytes
        
        if key in grid_history:  
            period = epoch - grid_history[key]
            if verbose:
                print( f'the global period is {period} epochs')
            return period, epoch
            
        
        grid_history[key] = epoch  # salva epoch, non la griglia intera
        grid = rule(grid)
    if verbose:
        print( f'there are no global repetition')
    return None

In [ ]:
#analyze the frequency of an oscillator pattern

pattern = rle_to_pattern('bakersdozenreactions')
plot_alive_cells(pattern, epochs=60, size=(50, 50), figsize=(18, 8), fps=5)
period= analyze_frequency(pattern, 100, (50, 50))

print(f"The pattern repeats after {period} epochs")

### Time to stability for a random pattern

In [ ]:
#tabella per categorizzare i pattern sulla base delle densità iniziali delle celle
def categorize_pattern(epochs, size_grid, n_sim, densities):

    results={ d :{'still_life' : 0,'oscillator':0, 'chaotic':0, 'extinction':0} for d in densities}

    for d in densities:
        for _ in range( n_sim):
            #generate a pattern 
            a=size_grid[0]*size_grid[1]*d
            pattern = np.zeros(size_grid[0]*size_grid[1], dtype=int)
            pattern[:int(a)] = 1
            npr.shuffle(pattern)
            pattern = pattern.reshape(size_grid)

            period, _= analyze_frequency(pattern, epochs, size_grid) 
            states= GoL_simulation(pattern, epochs, size_grid)
            occ = states[-1].sum() / states[-1].size
            
            if occ == 0:
                results[d]['extinction'] += 1      
            elif period is None:
                results[d]['chaotic'] += 1
            elif period == 1:
                results[d]['still_life'] += 1
            else:
                results[d]['oscillator'] += 1 

    rows= []
    for d, counts in results.items():
            rows.append({'density (%)' : round(d * 100),
                             'Still Life (%)'  : (counts['still_life']  / n_sim) * 100,
                            'Oscillator (%)'  : (counts['oscillator']  / n_sim) * 100,
                            'Chaotic (%)'     : (counts['chaotic']     / n_sim) * 100,
                            'Extinction (%)' : (counts['extinction']  / n_sim) * 100,})
    
    df = pd.DataFrame(rows).set_index('density (%)')

    return df

from the table above we can observe the distribution of the final configurations as a function of the initial cell's density. At low density, patterns tend to stabilize into still lifes or oscillators, with occasional extinctions. At intermediate densities, the system exhibits the richest variety of behaviors, consistent with the known critical regime of the Game of Life. Instead, at high densities the extinction rate increases sharply, reaching 90% at maximum density, as most cells die due to the overpopulation rule. Of course, the noise depend on the number of simulations, in our case n= 40. 

In [ ]:
def plot_distribution( epochs, size_grid, n_sim, densities):
    df= categorize_pattern( epochs, size_grid, n_sim, densities)
    plt.plot(df.index, df['Still Life (%)'], label='still_life', color='red')
    plt.plot(df.index, df['Oscillator (%)'], label='oscillator', color='green')
    plt.plot(df.index, df['Chaotic (%)'], label='chaotic', color='blue')
    plt.plot(df.index, df['Extinction (%)'], label='extinction', color='orange')
    for col, color in zip(df.columns, ['red','green','blue','orange']):
        p = df[col] / 100 #probabilità che esca un pattern 
        err = np.sqrt(p * (1 - p) / n_sim) * 100 #errore standard basato sulle sole 50 simulazioni
        plt.fill_between(df.index, df[col] - err, df[col] + err, color=color, alpha= 0.15)
    plt.xlabel( 'Initial density (%)')
    plt.ylabel( 'Probability of a pattern ( %)')
    plt.legend()
    plt.title( 'Pattern Distribution by Initial Density')
    plt.grid()
    plt.show()
    assert np.allclose( df.sum( axis=1), 100), "le categorie non sommano a 100"

In [ ]:
plot_distribution(500, ( 10, 10), 500, np.arange(0.05, 1, 0.05) )

The plot shows how the probability of the 4 pattern categories evolves with initial density, obtained from 500 independent simulations per point, a number sufficient to guarantee narrow error bands and statistically reliable estimates.
Three distinct regimes can be identified:
* Low density (5–15%): Extinction dominates, reaching a maximum probability at density 5%. Cells are too few to interact and die due to underpopulation. The transition toward the stable regime is very sharp, suggesting the existence of a critical survival density around 15%, where the cells begin to be close enough to interact with each other and form stable structures.
* Intermediate density (15–75%): Still Life and Oscillator share the domain roughly equally, at approximately 50% and 45% respectively. The error bands of the two categories overlap completely throughout this region, meaning the two categories are statistically indistinguishable: it is not the initial density that determines which one prevails, but rather the specific random configuration of the initial pattern. 
* High density (75–95%): Extinction rises sharply again, symmetrically to the low density regime. Overpopulation is just as lethal as isolation, due to the GoL rule that kills any cell with more than 3 neighbors. Both Still Life and Oscillator drop to zero, as there is no room for stable structures to form.

The following table reports the exact values at the most significant density points, corresponding to the transition regions and the plateau of the intermediate regime, that we observed from the plot.

In [ ]:
densities_table = [0.05, 0.15, 0.25, 0.50, 0.75, 0.85, 0.95]
df=categorize_pattern( 500, (20,20), 500, densities_table)
df

We can see how this plot change if we take a much large grid:

In [ ]:
plot_distribution(500, ( 20, 20), 500, np.arange(0.05, 1, 0.05) )

In [ ]:
plot_distribution(500, ( 50, 50), 500, np.arange(0.05, 1, 0.05) )

Increasing the grid size from 20×20 to 50×50 leads to dramatically change in the the intermediate regime: Still Life drops from ~50% to ~2%, while Oscillators rise to ~70% and Chaotic patterns appear at ~30%. This suggests that grid size acts as a "freedom parameter", larger grids allow dynamic structures to evolve freely without boundary interference, preventing stabilization into static configurations.


Ettore

In [ ]:
# Random pattern generator

size, epochs = (20, 20), 200
x = 0.3
a = size[0]*size[1]*x
pattern = np.zeros(size[0]*size[1], dtype=int)
pattern[:int(a)] = 1
npr.shuffle(pattern)
pattern = pattern.reshape(size)

plot_alive_cells(pattern, epochs, size, figsize=(18, 8))

Analisi Occupanza nel tempo Kevin (dice Ettore)

consiglio di citare nel discorso:
- discorso che al crescere della griglia c'è più prosperità (vedi percentuale) e variabilità (alta std)
- teorema del limte centrale che si perde con l'applicazione delle regole

In [ ]:
def occupancy(epochs, N_sim, size_grid=(20, 20), size_pattern=(20, 20)):

    occ = np.zeros((N_sim, epochs))
    
    for sim in range(N_sim):
        pattern = npr.randint(0, 2, size=size_pattern)   # random pattern
        states = GoL_simulation(pattern, epochs, size_grid)
        
        for t in range(epochs):
            occ[sim, t] = np.sum(states[t])
        
    return occ

def occupancy_plot(epochs, N_sim, size_grid, size_pattern):
    
    occ = occupancy(epochs, N_sim, size_grid, size_pattern)
    mean_occupancy = np.mean(occ, axis=0)
    std_occupancy = np.std(occ, axis=0)

    print(f"Mean Initial state density = {mean_occupancy[0]/(size_grid[0]*size_grid[1]):.3f}") 
    print(f"Mean Final state density = {mean_occupancy[epochs-1] / (size_grid[0]*size_grid[1]):.3f}")
    print(f"Relative decrease of the density = {(mean_occupancy[0] - mean_occupancy[epochs-1]) * 100 / mean_occupancy[0]:.3f} %")
    
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    t = np.arange(epochs)
    
    ax[0].plot(t, mean_occupancy, linewidth=2, label="Mean occupancy")
    ax[0].fill_between(t, mean_occupancy - std_occupancy, mean_occupancy + std_occupancy, alpha=0.3, label="±1 std")
    ax[0].set_xlabel("Epochs", fontsize=14)
    ax[0].set_ylabel("Mean occupancy", fontsize=14)
    ax[0].set_title("Average Occupancy vs Time", fontsize=18)
    ax[0].grid(True)
    ax[0].legend(fontsize=14)

    ax[1].hist(occ[:, 0], bins=int(np.sqrt(epochs)), density=True, color='r', label='Initial Occupancy')
    ax[1].hist(occ[:, epochs - 1], bins=int(np.sqrt(epochs)), density=True, label='Final Occupancy')
    ax[1].set_xlabel("Number of alive cells", fontsize=14)
    ax[1].set_ylabel("Probability density", fontsize=14)
    ax[1].set_title("Distribution of Occupancy: Initial vs Final", fontsize=18)
    ax[1].grid(True)
    ax[1].legend(fontsize=14)

    plt.tight_layout()
    plt.show()

In [ ]:
occupancy_plot(500, 500, (10, 10), (10, 10))

In [ ]:
occupancy_plot(500, 500, (20, 20), (10, 10)) 

In [ ]:
occupancy_plot(500, 500, (50, 50), (10, 10)) 

## References

- Johnston, N., & Greene, D. (2022).  
  *Conway's Game of Life: Mathematics and Construction*.  
  Available at: https://conwaylife.com/book/